# Reproduce goal representations and steering in Procgen Maze

**Paper:** [Understanding and Controlling a Maze-Solving Policy Network](https://arxiv.org/abs/2310.08043v1) (arXiv v1, 12 October 2023)  
**Reference code:** [`UlisseMini/procgen-tools@dc2243c99110aca92687cdf566daafcfbe7067a0`](https://github.com/UlisseMini/procgen-tools/tree/dc2243c99110aca92687cdf566daafcfbe7067a0)  
**Target policy:** `maze_I/model_rand_region_5.pth`, IMPALA scale 15, `procgen==0.10.7`  
**Target layer:** `embedder.block2.res1.resadd_out`  
**Default execution:** bounded synthetic smoke path, not a scientific reproduction  
**Current evidence:** adapter parity, representation/control metrics, matched intervention mechanics, and provenance only; paper results are **not evaluated**.

## Asset availability boundary

The versioned paper PDF has SHA-256 `4b18fece611f794b8bce5583351e86288ae01dbb7cfbc6f440b106ef61658d1a`. The reference repository is MIT-licensed and pins Procgen 0.10.7, but its advertised `https://nerdsniper.net/mats/model_rand_region_5.pth` download returns 404. The linked Google Drive folder is mutable, and the checkpoint has no separately declared digest or model-artifact license. The paper does not publish an immutable evaluation-maze manifest or reference metric bundle.

Scientific mode therefore fails closed. It never substitutes another checkpoint, a freshly downloaded mutable file, or a merely compatible Procgen build. Set `XDRL_MAZE_POLICY_MODE=paper` only after this notebook is updated with a verified checkpoint digest and license, a frozen level manifest, and reference outputs. Until then, all scientific claims remain blocked.

## Frozen protocol and claim boundary

The paper/reference-code channel set is predeclared as `[7, 8, 42, 44, 55, 77, 82, 88, 89, 99, 113]`; its more behaviorally effective subset is `[8, 55, 77, 82, 88, 89, 113]`. Representation recovery is evaluated on levels held out before any metric is computed. Reported-channel localization is compared with a seeded shuffled-channel mask and shuffled goal labels. Single-channel, prefix-size, and leave-one-out results test redundancy/distribution without redefining the selected channels after seeing held-out outcomes. Level bootstrap intervals quantify uncertainty.

The steering target, magnitude, layer, effective channels, action order, level split, and controls are frozen below. Every vector-field arm shares one checkpoint, complete observation tensor, level IDs, agent positions, and random seed. The baseline is an explicit no-op; controls patch a seeded shuffled channel mask or the effective channels at a spatially shuffled target. Report policy-distribution changes separately from greedy-rollout behavior.

XDRL owns the typed interaction contract, execution boundary, matched workflow, and artifact identities. TDHook owns the public intervention workflow. No `procgen-tools` hook cache or private hook state is used. These manifests establish matched mechanics and provenance, not causality. Correlation/localization, causal contribution, controllability, and claims about the policy's goals remain distinct. The smoke fixture is not Procgen, not the paper checkpoint, and not evidence about learned policy goals.

In [ ]:
import copy
import hashlib
import json
import os
import subprocess

import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import SteeringVectors
from tdhook.workflow import Workflow
from xdrl import (
    ArtifactDigestAlgorithm,
    BatchSemantics,
    InputArtifactReference,
    InputArtifactRole,
    InteractionContract,
    InteractionPhase,
    KeyPresence,
    KeyRole,
    KeySchema,
    ModelRole,
    OutputArtifactDeclaration,
    OutputArtifactDigest,
    OutputArtifactRole,
    RuntimeInteractionContext,
    TDHookWorkflowRunner,
    TensorDictSchema,
)

MODE = os.environ.get("XDRL_MAZE_POLICY_MODE", "smoke")
SEED = 5801
HEIGHT = WIDTH = 6
ACTION_ORDER = ("UP", "DOWN", "LEFT", "RIGHT")
ACTION_DELTA = torch.tensor(((-1, 0), (1, 0), (0, -1), (0, 1)))
REFERENCE_REVISION = "dc2243c99110aca92687cdf566daafcfbe7067a0"
PAPER_PDF_SHA256 = "4b18fece611f794b8bce5583351e86288ae01dbb7cfbc6f440b106ef61658d1a"
PAPER_CHECKPOINT_RELEASE = None
PROCGEN_VERSION = "0.10.7"
PAPER_MAZE_SEEDS = tuple(range(100))
TARGET_LAYER = "embedder.block2.res1.resadd_out"
WORKFLOW_TARGET = f"module.{TARGET_LAYER}"
REPORTED_CHANNELS = (7, 8, 42, 44, 55, 77, 82, 88, 89, 99, 113)
EFFECTIVE_CHANNELS = (8, 55, 77, 82, 88, 89, 113)
INTERVENTION_TARGET = (4, 1)
SPATIALLY_SHUFFLED_TARGET = (1, 4)
INTERVENTION_MAGNITUDE = 5.5


def repository_revision():
    revision = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
    dirty = subprocess.run(
        ["git", "status", "--porcelain", "--untracked-files=no"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    return f"{revision}+dirty" if dirty else revision


CODE_REVISION = repository_revision()

if MODE not in {"smoke", "paper"}:
    raise ValueError("XDRL_MAZE_POLICY_MODE must be 'smoke' or 'paper'")
if MODE == "paper" and PAPER_CHECKPOINT_RELEASE is None:
    raise RuntimeError(
        "paper mode is blocked: the target checkpoint lacks an immutable digest/license and its advertised URL is unavailable; "
        "a frozen maze manifest and reference outputs are also required"
    )

_ = torch.manual_seed(SEED)

## Build a bounded frozen maze fixture

Smoke levels are open 6x6 grids with one agent and one goal. Their IDs, goal coordinates, every vector-field agent position, and the train/evaluation split are deterministic. The fixture deliberately places a spatial goal map in the eleven predeclared channels, allowing the analysis to detect a known signal and the controls to fail as expected.

In [ ]:
def smoke_bundle(level_count=32):
    level_ids = torch.arange(level_count)
    positions = torch.cartesian_prod(torch.arange(HEIGHT), torch.arange(WIDTH))
    goals = torch.stack(((2 * level_ids + 1) % HEIGHT, (3 * level_ids + 2) % WIDTH), dim=-1)
    observations = torch.zeros(level_count, len(positions), 3, HEIGHT, WIDTH)
    observations[:, :, 0] = 1.0
    for level in range(level_count):
        observations[level, :, 2, goals[level, 0], goals[level, 1]] = 1.0
        for position_index, (row, column) in enumerate(positions):
            observations[level, position_index, 1, row, column] = 1.0
    return {
        "environment": "synthetic-open-maze-smoke-v1",
        "level_ids": level_ids,
        "positions": positions,
        "goals": goals,
        "observations": observations,
    }


bundle = smoke_bundle()
train_levels = torch.arange(0, 24)
evaluation_levels = torch.arange(24, 32)
assert not set(train_levels.tolist()) & set(evaluation_levels.tolist())

channel_generator = torch.Generator().manual_seed(SEED + 17)
channel_permutation = torch.randperm(128, generator=channel_generator).tolist()
SHUFFLED_CHANNELS = tuple(channel for channel in channel_permutation if channel not in REPORTED_CHANNELS)[:11]

{
    "mode": MODE,
    "environment": bundle["environment"],
    "train_level_ids": bundle["level_ids"][train_levels].tolist(),
    "evaluation_level_ids": bundle["level_ids"][evaluation_levels].tolist(),
    "reported_channels": REPORTED_CHANNELS,
    "shuffled_channel_control": SHUFFLED_CHANNELS,
    "scientific_claim_ready": False,
}

## Declare the adapter and verify uninstrumented parity

The adapter mapping is explicit: RGB-like fixture observations map to `observation`; the reference layer maps to `goal_features`; action logits use `(UP, DOWN, LEFT, RIGHT)`. A real paper run must implement the original 15-scale IMPALA adapter, load the exact checkpoint unchanged, compare every parameter tensor, and demonstrate output parity against the pinned reference policy before hooks are installed.

In [ ]:
class ResidualOne(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.resadd_out = torch.nn.Identity()

    def forward(self, value):
        return self.resadd_out(value)


class BlockTwo(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.res1 = ResidualOne()

    def forward(self, value):
        return self.res1(value)


class SmokeEmbedder(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = torch.nn.Conv2d(3, 128, 1, bias=False)
        self.block2 = BlockTwo()
        torch.nn.init.zeros_(self.projection.weight)
        with torch.no_grad():
            self.projection.weight[:, 0, 0, 0] = torch.linspace(-0.03, 0.03, 128)
            self.projection.weight[:, 1, 0, 0] = torch.linspace(0.02, -0.02, 128)
            for offset, channel in enumerate(REPORTED_CHANNELS):
                self.projection.weight[channel, 2, 0, 0] = 0.8 + 0.03 * offset

    def forward(self, observation):
        return self.block2(self.projection(observation))


class TinyMazePolicy(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.embedder = SmokeEmbedder()

    def forward(self, observation):
        features = self.embedder(observation)
        goal_map = features[:, REPORTED_CHANNELS].mean(dim=1)
        goal_weight = torch.softmax(20.0 * goal_map.flatten(1), dim=-1)
        rows = torch.arange(HEIGHT, dtype=features.dtype, device=features.device).repeat_interleave(WIDTH)
        columns = torch.arange(WIDTH, dtype=features.dtype, device=features.device).repeat(HEIGHT)
        goal_row = (goal_weight * rows).sum(-1)
        goal_column = (goal_weight * columns).sum(-1)
        agent = observation[:, 1]
        agent_row = (agent.sum(-1) * torch.arange(HEIGHT, dtype=features.dtype, device=features.device)).sum(-1)
        agent_column = (agent.sum(-2) * torch.arange(WIDTH, dtype=features.dtype, device=features.device)).sum(-1)
        row_delta = goal_row - agent_row
        column_delta = goal_column - agent_column
        logits = 4.0 * torch.stack((-row_delta, row_delta, -column_delta, column_delta), dim=-1)
        return features, logits


reference_policy = TinyMazePolicy().eval()
adapted_core = copy.deepcopy(reference_policy).eval()
policy = TensorDictModule(adapted_core, in_keys=["observation"], out_keys=["goal_features", "logits"])
flat_observation = bundle["observations"].flatten(0, 1)
batch = TensorDict({"observation": flat_observation}, batch_size=[len(flat_observation)])
batch_semantics = BatchSemantics(("transition",))
contract = InteractionContract(
    identity="maze-goal-representations:synthetic-policy:evaluation",
    role=ModelRole.ACTOR,
    phase=InteractionPhase.EVALUATION,
    module_path="policy",
    input_schema=TensorDictSchema(
        (KeySchema("observation", KeyRole.OBSERVATION, KeyPresence.REQUIRED),), batch_semantics
    ),
    output_schema=TensorDictSchema(
        (
            KeySchema("goal_features", KeyRole.FEATURE, KeyPresence.PRODUCED),
            KeySchema("logits", KeyRole.ACTION, KeyPresence.PRODUCED),
        ),
        batch_semantics,
    ),
    model_id="tiny-maze-goal-map-smoke",
    checkpoint_id="synthetic:seed-5801",
    module_training=False,
)
interaction = RuntimeInteractionContext(contract, policy, batch)
with torch.inference_mode():
    reference_features, reference_logits = reference_policy(flat_observation)
    adapted_output = interaction(batch.clone())

parameter_parity = all(
    torch.equal(reference_policy.state_dict()[name], adapted_core.state_dict()[name])
    for name in reference_policy.state_dict()
)
output_parity = torch.equal(reference_features, adapted_output["goal_features"]) and torch.equal(
    reference_logits, adapted_output["logits"]
)
assert parameter_parity and output_parity
features = adapted_output["goal_features"].reshape(32, HEIGHT * WIDTH, 128, HEIGHT, WIDTH)
baseline_logits = adapted_output["logits"].reshape(32, HEIGHT * WIDTH, len(ACTION_ORDER))
{
    "adapter_mapping": {
        "reference_input": "RGB observation -> observation",
        "reference_layer": f"{TARGET_LAYER} -> goal_features",
        "reference_actions": ACTION_ORDER,
    },
    "smoke_parameter_parity": parameter_parity,
    "smoke_uninstrumented_output_parity": output_parity,
    "paper_parameter_parity": "not evaluated",
    "paper_uninstrumented_output_parity": "not evaluated",
}

## Recover the predeclared channel representation with controls

Localization predicts the goal square from the maximum of the mean selected-channel map. One activation map per level is used so the repeated agent-position vector-field batch is not mistaken for independent level evidence. The held-out labels are untouched except in the explicitly named shuffled-label control.

In [ ]:
level_features = features[:, 0]


def predicted_coordinate(feature_maps, channels):
    score = feature_maps[:, channels].mean(dim=1).flatten(1)
    flat = score.argmax(-1)
    return torch.stack((flat // WIDTH, flat % WIDTH), dim=-1)


def coordinate_accuracy(prediction, target):
    return float((prediction == target).all(dim=-1).float().mean())


def level_bootstrap_accuracy(prediction, target, draws=400):
    generator = torch.Generator().manual_seed(SEED + 101)
    values = []
    for _ in range(draws):
        indices = torch.randint(len(target), (len(target),), generator=generator)
        values.append(coordinate_accuracy(prediction[indices], target[indices]))
    return [float(value) for value in torch.tensor(values).quantile(torch.tensor((0.025, 0.975)))]


eval_features = level_features[evaluation_levels]
eval_goals = bundle["goals"][evaluation_levels]
reported_prediction = predicted_coordinate(eval_features, REPORTED_CHANNELS)
shuffled_channel_prediction = predicted_coordinate(eval_features, SHUFFLED_CHANNELS)
shuffled_goals = eval_goals.roll(1, dims=0)

single_channel_accuracy = {
    channel: coordinate_accuracy(predicted_coordinate(eval_features, (channel,)), eval_goals)
    for channel in REPORTED_CHANNELS
}
prefix_accuracy = {
    count: coordinate_accuracy(predicted_coordinate(eval_features, REPORTED_CHANNELS[:count]), eval_goals)
    for count in (1, 3, 7, 11)
}
leave_one_out_accuracy = {
    removed: coordinate_accuracy(
        predicted_coordinate(eval_features, tuple(channel for channel in REPORTED_CHANNELS if channel != removed)),
        eval_goals,
    )
    for removed in REPORTED_CHANNELS
}
representation_results = {
    "reported_channels": {
        "accuracy": coordinate_accuracy(reported_prediction, eval_goals),
        "level_bootstrap_95": level_bootstrap_accuracy(reported_prediction, eval_goals),
    },
    "seeded_shuffled_channel_mask": {
        "channels": SHUFFLED_CHANNELS,
        "accuracy": coordinate_accuracy(shuffled_channel_prediction, eval_goals),
    },
    "shuffled_goal_labels": coordinate_accuracy(reported_prediction, shuffled_goals),
    "single_channel_accuracy": single_channel_accuracy,
    "prefix_accuracy": prefix_accuracy,
    "leave_one_out_accuracy": leave_one_out_accuracy,
}
representation_results

## Run matched vector-field steering arms

The reference notebooks set selected pixels directly; the smoke intervention mirrors that mechanism at the declared layer. `reported_effective` patches the seven predeclared effective channels at `(4, 1)`. `shuffled_channels` uses the same count, magnitude, and target with a seeded non-reported mask. `spatially_shuffled_target` uses the effective channels and magnitude at `(1, 4)`. Each arm is paired with the same explicit no-op baseline through the public XDRL/TDHook workflow.

In [ ]:
def tensor_digest(tensor):
    value = tensor.detach().cpu().contiguous()
    header = json.dumps({"dtype": str(value.dtype), "shape": list(value.shape)}, sort_keys=True).encode()
    return hashlib.sha256(header + value.numpy().tobytes()).hexdigest()


def named_tensor_digest(named_tensors):
    manifest = [{"name": name, "sha256": tensor_digest(tensor)} for name, tensor in sorted(named_tensors)]
    return hashlib.sha256(json.dumps(manifest, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def module_digest(module):
    return named_tensor_digest(module.state_dict().items())


def no_op(*, output, **_):
    return output


def patch_at(channels, coordinate):
    channels = tuple(channels)
    coordinate = tuple(coordinate)

    def callback(*, output, **_):
        edited = output.clone()
        edited[:, channels, coordinate[0], coordinate[1]] = INTERVENTION_MAGNITUDE
        return edited

    return callback


FROZEN_LEVEL_TENSORS = (
    ("level_ids", bundle["level_ids"]),
    ("positions", bundle["positions"]),
    ("goals", bundle["goals"]),
    ("observations", bundle["observations"]),
)
common_inputs = (
    InputArtifactReference(
        "checkpoint:tiny-maze-goal-map-smoke",
        InputArtifactRole.MODEL_CHECKPOINT,
        ArtifactDigestAlgorithm.SHA256,
        module_digest(adapted_core),
        metadata={"mode": MODE, "paper_checkpoint": False, "action_order": list(ACTION_ORDER)},
    ),
    InputArtifactReference(
        "levels:synthetic-open-maze-smoke-v1",
        InputArtifactRole.LEVEL_SET,
        ArtifactDigestAlgorithm.SHA256,
        named_tensor_digest(FROZEN_LEVEL_TENSORS),
        metadata={
            "level_ids": bundle["level_ids"].tolist(),
            "evaluation_level_ids": bundle["level_ids"][evaluation_levels].tolist(),
            "frozen": True,
        },
    ),
    InputArtifactReference(
        "paper:arxiv-2310.08043v1",
        InputArtifactRole.OTHER,
        ArtifactDigestAlgorithm.SHA256,
        PAPER_PDF_SHA256,
        source="https://arxiv.org/pdf/2310.08043v1",
        revision="arXiv:2310.08043v1",
        source_is_immutable=True,
        metadata={
            "reference_code_revision": REFERENCE_REVISION,
            "procgen_version": PROCGEN_VERSION,
            "checkpoint_available": False,
            "checkpoint_license": "not separately declared",
            "artifact_kind": "paper_pdf",
            "reference_results_available": False,
        },
    ),
)


def result_resolver(data, declarations):
    digest = named_tensor_digest((("goal_features", data["goal_features"]), ("logits", data["logits"])))
    return tuple(OutputArtifactDigest(item.identity, ArtifactDigestAlgorithm.SHA256, digest) for item in declarations)


runner = TDHookWorkflowRunner(RuntimeInteractionContext(contract, policy, batch))
intervention_arms = {
    "reported_effective": (EFFECTIVE_CHANNELS, INTERVENTION_TARGET),
    "shuffled_channels": (SHUFFLED_CHANNELS[: len(EFFECTIVE_CHANNELS)], INTERVENTION_TARGET),
    "spatially_shuffled_target": (EFFECTIVE_CHANNELS, SPATIALLY_SHUFFLED_TARGET),
}


def run_arm(label, channels, coordinate):
    callback = patch_at(channels, coordinate)
    pair = runner.run_paired(
        Workflow(SteeringVectors([WORKFLOW_TARGET], steer_fn=no_op)),
        Workflow(SteeringVectors([WORKFLOW_TARGET], steer_fn=callback)),
        batch,
        pair_id=f"maze-goal-representations:smoke:{label}",
        code_revision=CODE_REVISION,
        declared_workflow_differences=(0,),
        seed=SEED,
        input_artifacts=common_inputs,
        baseline_output_artifacts=(
            OutputArtifactDeclaration(f"result:{label}:no-op", OutputArtifactRole.INTERVENTION_RESULT),
        ),
        baseline_output_artifact_resolver=result_resolver,
        intervention_output_artifacts=(
            OutputArtifactDeclaration(f"result:{label}:patched", OutputArtifactRole.INTERVENTION_RESULT),
        ),
        intervention_output_artifact_resolver=result_resolver,
        callback_identifiers={
            no_op: "no-op",
            callback: f"set-channels-{list(channels)}-at-{tuple(coordinate)}-to-{INTERVENTION_MAGNITUDE}",
        },
    )
    assert pair.manifest.interpretation == "mechanics_and_provenance_only"
    assert pair.baseline.provenance.input_artifacts == pair.intervention.provenance.input_artifacts
    return pair


pairs = {label: run_arm(label, *specification) for label, specification in intervention_arms.items()}
{label: pair.manifest.interpretation for label, pair in pairs.items()}

## Quantify policy-distribution and behavioural effects

Vector-field probabilities cover every agent square in every frozen level. Distribution metrics are mean total variation, baseline-to-intervention KL, greedy-action change rate, and the probability change for an action that moves toward the declared steering target. Behaviour is a deterministic greedy rollout on the open fixture; success at the real goal and the steering target are reported independently, with paired level-bootstrap uncertainty for success-rate changes.

In [ ]:
def desired_action(positions, target):
    row_delta = target[0] - positions[:, 0]
    column_delta = target[1] - positions[:, 1]
    action = torch.where(
        row_delta < 0,
        torch.zeros_like(row_delta),
        torch.where(
            row_delta > 0,
            torch.ones_like(row_delta),
            torch.where(column_delta < 0, torch.full_like(row_delta, 2), torch.full_like(row_delta, 3)),
        ),
    )
    valid = (row_delta != 0) | (column_delta != 0)
    return action, valid


def greedy_success_by_level(probabilities, targets, horizon=12):
    probabilities = probabilities.reshape(32, HEIGHT * WIDTH, len(ACTION_ORDER))
    successes = []
    for level in range(32):
        target = targets[level] if targets.ndim == 2 else targets
        level_results = []
        for start in bundle["positions"]:
            position = start.clone()
            reached = bool(torch.equal(position, target))
            for _ in range(horizon):
                if reached:
                    break
                flat_index = int(position[0] * WIDTH + position[1])
                action = int(probabilities[level, flat_index].argmax())
                position = (position + ACTION_DELTA[action]).clamp(
                    torch.tensor((0, 0)), torch.tensor((HEIGHT - 1, WIDTH - 1))
                )
                reached = bool(torch.equal(position, target))
            level_results.append(reached)
        successes.append(torch.tensor(level_results, dtype=torch.float32).mean())
    return torch.stack(successes)


def paired_bootstrap_delta(baseline, changed, draws=400):
    generator = torch.Generator().manual_seed(SEED + 303)
    values = []
    for _ in range(draws):
        indices = torch.randint(len(evaluation_levels), (len(evaluation_levels),), generator=generator)
        values.append(float((changed[indices] - baseline[indices]).mean()))
    return [float(value) for value in torch.tensor(values).quantile(torch.tensor((0.025, 0.975)))]


effect_results = {}
for label, pair in pairs.items():
    _channels, arm_target = intervention_arms[label]
    target_actions, target_mask = desired_action(bundle["positions"], arm_target)
    target_actions = target_actions.repeat(len(evaluation_levels))
    target_mask = target_mask.repeat(len(evaluation_levels))
    fixed_steering_targets = torch.tensor(arm_target).repeat(32, 1)
    baseline_probability = pair.baseline.data["logits"].softmax(-1).reshape(32, HEIGHT * WIDTH, -1)
    changed_probability = pair.intervention.data["logits"].softmax(-1).reshape(32, HEIGHT * WIDTH, -1)
    evaluation_baseline_probability = baseline_probability[evaluation_levels].flatten(0, 1)
    evaluation_changed_probability = changed_probability[evaluation_levels].flatten(0, 1)
    baseline_goal_success = greedy_success_by_level(baseline_probability, bundle["goals"])
    changed_goal_success = greedy_success_by_level(changed_probability, bundle["goals"])
    baseline_target_success = greedy_success_by_level(baseline_probability, fixed_steering_targets)
    changed_target_success = greedy_success_by_level(changed_probability, fixed_steering_targets)
    baseline_target_probability = evaluation_baseline_probability.gather(-1, target_actions[:, None]).squeeze(-1)
    changed_target_probability = evaluation_changed_probability.gather(-1, target_actions[:, None]).squeeze(-1)
    effect_results[label] = {
        "arm_target": list(arm_target),
        "mean_total_variation": float(
            0.5 * (evaluation_changed_probability - evaluation_baseline_probability).abs().sum(-1).mean()
        ),
        "mean_kl_baseline_to_changed": float(
            torch.nn.functional.kl_div(
                evaluation_changed_probability.log(), evaluation_baseline_probability, reduction="batchmean"
            )
        ),
        "greedy_action_change_rate": float(
            (evaluation_changed_probability.argmax(-1) != evaluation_baseline_probability.argmax(-1)).float().mean()
        ),
        "target_directed_probability_delta": float(
            (changed_target_probability[target_mask] - baseline_target_probability[target_mask]).mean()
        ),
        "real_goal_success": {
            "baseline": float(baseline_goal_success[evaluation_levels].mean()),
            "intervention": float(changed_goal_success[evaluation_levels].mean()),
            "paired_level_bootstrap_delta_95": paired_bootstrap_delta(
                baseline_goal_success[evaluation_levels], changed_goal_success[evaluation_levels]
            ),
        },
        "steering_target_success": {
            "baseline": float(baseline_target_success[evaluation_levels].mean()),
            "intervention": float(changed_target_success[evaluation_levels].mean()),
            "paired_level_bootstrap_delta_95": paired_bootstrap_delta(
                baseline_target_success[evaluation_levels], changed_target_success[evaluation_levels]
            ),
        },
    }

effect_results

## Evidence verdict

The final object separates runtime success, asset gates, reference agreement, empirical smoke metrics, and claim readiness. Positive smoke results cannot upgrade an unavailable checkpoint into a reproduction. Negative controls and failed scientific gates remain visible. A future paper-exact execution must preserve the frozen protocol, validate the real adapter against the reference implementation, record exact checkpoint/environment/level identities, and compare a reference metric bundle before any scientific field can change.

In [ ]:
verdict = {
    "smoke_execution": "passed",
    "runtime_code_revision": CODE_REVISION,
    "paper_exact_assets": {
        "status": "blocked",
        "paper": {"revision": "arXiv:2310.08043v1", "sha256": PAPER_PDF_SHA256},
        "reference_code": {"revision": REFERENCE_REVISION, "license": "MIT"},
        "checkpoint": {
            "identity": "maze_I/model_rand_region_5.pth",
            "advertised_download": "unavailable (HTTP 404)",
            "digest": "not published",
            "license": "not separately declared",
        },
        "procgen_version": PROCGEN_VERSION,
        "paper_maze_seed_target": list(PAPER_MAZE_SEEDS),
        "frozen_paper_level_manifest": "not published",
    },
    "adapter_parity": {
        "smoke_parameter_parity": parameter_parity,
        "smoke_uninstrumented_output_parity": output_parity,
        "paper_reference_parity": "not evaluated",
    },
    "goal_representation": {
        "status": "smoke-only known-signal recovery",
        "metrics": representation_results,
        "correlation_claim": "not evaluated on the paper checkpoint",
    },
    "steering": {
        "status": "smoke-only controllability mechanics",
        "effects": effect_results,
        "matched_provenance": {label: pair.manifest.to_dict() for label, pair in pairs.items()},
    },
    "causal_contribution": "not established",
    "policy_goal_claim": "blocked",
    "reference_agreement": "not evaluated",
    "material_deviations": [
        "synthetic open-grid observations replace Procgen Maze only in smoke mode",
        "a deterministic goal-map fixture replaces the unavailable IMPALA checkpoint only in smoke mode",
        "greedy vector-field rollouts replace Procgen environment rollouts only in smoke mode",
        "no paper metric bundle is available for numerical agreement",
    ],
    "claim_limit": "No inference to the paper checkpoint, other checkpoints, Procgen mazes, or learned policy goals.",
    "scientific_claim_ready": False,
}
verdict